# Using co-occurrences to find describing adjectives near Character Names

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import spacy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests
from collections import defaultdict
from tqdm.notebook import tqdm_notebook
from IPython.display import Markdown, display

tqdm_notebook.pandas()

In [ ]:
BATCH_SIZE = 64
N_PROCESS = 4

#### Import gender, speakers and scripts

In [ ]:
# load data
gender_df = pd.read_csv("../data/gender/gender.csv", encoding="utf-8")
print("loaded genders of", len(gender_df), "speakers")
scriptdata = pd.read_csv("../combine/combine.csv", encoding="utf-8")
print("loaded", len(scriptdata), "script lines")
speakers = pd.read_csv("../data/rolle.tsv", encoding="utf-8", delimiter="\t")
print("loaded", len(speakers), "speakers")
additional_speakers = pd.read_csv("../data/additional_roles.csv", encoding="utf-8")
print("loaded", len(additional_speakers), "additional_speakers")

In [ ]:
speakers = pd.concat(
    [speakers, additional_speakers], ignore_index=True
).drop_duplicates(subset=["name"], keep="first")
speakers

#### Filter empty text lines and REGIE

In [ ]:
scriptdata = scriptdata[~scriptdata["Text"].isna()]
scriptdata = scriptdata[scriptdata["speaker_id"] != 0]
scriptdata = scriptdata.sample(n=3000, random_state=1)
scriptdata

#### Split texts into sentences

In [ ]:
nlp = spacy.load("de_core_news_lg")
nlp.add_pipe("sentencizer")

texts = scriptdata["Text"].tolist()
sentences = []

for doc in tqdm_notebook(
    nlp.pipe(texts, batch_size=BATCH_SIZE, n_process=N_PROCESS), total=len(texts)
):
    sentences.append([sent.text.strip() for sent in doc.sents])

scriptdata["sentences"] = sentences
scriptdata = scriptdata.explode("sentences")
scriptdata = scriptdata[~scriptdata["sentences"].isna()]
scriptdata = scriptdata.drop(columns=["Text"])
scriptdata

#### Build stopword set (use case-insensitive lower names)

In [ ]:
stopwords = set()
with open("../data/german_stopwords_full.txt", "r", encoding="utf-8") as f:
    stopwords.update([w.lower() for w in f.read().splitlines()])
with open("../clean_lemmatize/ext_stopwords-de.txt", "r", encoding="utf-8") as f:
    stopwords.update([w.lower() for w in f.read().splitlines()])
print("Created stopword set with", len(stopwords), "entries")

In [ ]:
list(stopwords)[:10]

#### Create a name to gender lookup

In [ ]:
# join speaker and gender dataframes
# inner join makes sure we only keep speakers with a known gender
speakers_with_gender = speakers.merge(gender_df, on="rolleID", how="inner")
speakers_with_gender.loc[-1] = [13, "Tante", "f"]  # adding a row
speakers_with_gender.index = speakers_with_gender.index + 1  # shifting index
speakers_with_gender = speakers_with_gender.sort_index()  # sorting by index

# create a speaker name to gender lookup
name_to_gender = dict(
    zip(
        speakers_with_gender["name"].str.lower().str.strip(),
        speakers_with_gender["gender"],
    )
)

# create character name set
character_name_set = set(name_to_gender.keys())
print(f"Lookup Set contains {len(character_name_set)} characters")

In [ ]:
speakers_with_gender[speakers_with_gender["rolleID"] == 13]

#### filter out sentences that do NOT contain character names

In [ ]:
# create regex pattern to find character names (case insensitive)
name_pattern = re.compile(
    "|".join(re.escape(name) for name in character_name_set), re.IGNORECASE
)

# copy/filter sentences that contain character names
character_rows = scriptdata[
    scriptdata["sentences"].notna()
    & scriptdata["sentences"].str.contains(name_pattern, regex=True)
].copy()

print(
    f"found {len(character_rows)} lines containing character names out of {len(scriptdata)} total"
)

#### Find character name - adjective pairs

In [ ]:
reverse_lemma_lookup = defaultdict(set)


def get_name_adj_pairs(doc):
    """
    For every token matching a character name, collect all ADJ tokens
    within +/- window positions.
    """

    adjectives = []
    character_names = []

    tokens = [
        token
        for token in list(doc)
        if not token.is_punct and not token.is_digit and not token.is_space
    ]
    # print(tokens)

    for i, tok in enumerate(tokens):
        tok_lower = tok.text.lower()
        if tok_lower in character_name_set:
            # save character name and position in doc
            character_names.append((tok_lower, i))
            # print(f'Found character name {tok_lower} in tokens {tokens}')
        elif tok.pos_ == "ADJ":
            # save adjectives lemma and position in doc
            adjectives.append((tok.lemma_.lower(), i))
            # append to reverse lookup, we can use this later to find the original form of the lemmatised token
            reverse_lemma_lookup[tok.lemma_.lower()].add(tok.text)
            # print(f'Found adj {tok.lemma_.lower()} in tokens {tokens}')

    pairs = []
    for name, name_idx in character_names:
        for adj, adj_idx in adjectives:
            pairs.append((name, adj))
            # TODO for future work: use np.abs(name_idx-adj_idx) to add information about the adj-name distance
    return pairs

In [ ]:
nlp = spacy.load("de_core_news_lg")
name_adj_pairs = []

sentences = character_rows["sentences"].tolist()
for doc in tqdm_notebook(
    nlp.pipe(sentences, batch_size=BATCH_SIZE, n_process=N_PROCESS),
    total=len(sentences),
):
    name_adj_pairs.extend(get_name_adj_pairs(doc))

print(name_adj_pairs[:10])
print(f"{len(name_adj_pairs)} (name, adj) pairs found")
print(f"filled reverse lemma lookup with {len(reverse_lemma_lookup)} entries")

#### Example for reverse lemma lookup

In [ ]:
print(
    f"Example for reverse lemma lookup: lemmas for adjective {name_adj_pairs[0][1]}: {reverse_lemma_lookup[name_adj_pairs[0][1]]}"
)

#### Create dataframe for statistical tests
- count name-adj combination
- assign gender through lookup
- create df

In [ ]:
name_adj_count_list = []
pair_counts = Counter(name_adj_pairs)

for (name, adj), count in pair_counts.items():
    gender = name_to_gender.get(
        name, "unknown"
    )  # unknown should not occur, but better safe than sorry
    name_adj_count_list.append((name, gender, adj, count))

name_adj_count_df = pd.DataFrame(
    name_adj_count_list, columns=["name", "gender", "adjective", "count"]
)
name_adj_count_df = name_adj_count_df.sort_values("count", ascending=False)
name_adj_count_df

- k   = total number of (name, adj) pairs across all sentences
- ki  = number of times name i appears (in any pair)
- kj  = number of times adjective j appears (in any pair)
- kij = number of times this specific (name, adj) pair appears

In [ ]:
def log_likelihood(kij, ki, kj, k):
    # safe log: avoid log(0)
    def xlogx(x):
        return x * np.log(x) if x > 0 else 0

    return 2 * (
        xlogx(k)
        - xlogx(ki)
        - xlogx(kj)
        + xlogx(kij)
        + xlogx(k - ki - kj + kij)
        + xlogx(ki - kij)
        + xlogx(kj - kij)
        - xlogx(k - ki)
        - xlogx(k - kj)
    )


def calculate_ll(row):
    return log_likelihood(row["count"], row["ki"], row["kj"], k)


# k = total number of (name, adj) pairs across all sentences
k = name_adj_count_df["count"].sum()  # total occurrences, not unique pairs

# sum of all name counts -> use for calculating ki
name_counts = name_adj_count_df.groupby("name")["count"].sum()

# sum of all adj counts -> use for calculating kj
adj_counts = name_adj_count_df.groupby("adjective")["count"].sum()

# create lookup in dataframe to make calculation faster (helper columns)
# for every row name/adj -> look up the counts and insert as new col
name_adj_count_df["ki"] = name_adj_count_df["name"].map(name_counts)
name_adj_count_df["kj"] = name_adj_count_df["adjective"].map(adj_counts)

# calculate log, likelihood in each row
name_adj_count_df["log_likelihood"] = name_adj_count_df.progress_apply(
    calculate_ll, axis=1
)

# drop helper columns
name_adj_count_df = name_adj_count_df.drop(columns=["ki", "kj"])
name_adj_count_df = name_adj_count_df.sort_values("log_likelihood", ascending=False)

name_adj_count_df

#### Adjust for multiple testing error using false discovery rate (fdr) with Benjamini/Hochberg (non-negative)

In [ ]:
# use chi-squared to calculate p-value from log-likelihood
name_adj_count_df["p_value"] = chi2.sf(name_adj_count_df["log_likelihood"], df=1)

# correct p value using benjamini-hochberg fdr-correction with default alpha of 0.05
reject, p_adjusted, _, _ = multipletests(name_adj_count_df["p_value"], method="fdr_bh")
name_adj_count_df["p_adjusted"] = p_adjusted
name_adj_count_df["significant"] = reject

name_adj_count_df

---

#### Until now, we looked at names and adjectives, now lets look at Gender

In [ ]:
male_name_adj_count_df = name_adj_count_df[name_adj_count_df["gender"] == "m"]
female_name_adj_count_df = name_adj_count_df[name_adj_count_df["gender"] == "f"]
intersect_f_m_adj = set(male_name_adj_count_df["adjective"]) | set(
    female_name_adj_count_df["adjective"]
)
m_total_cooc = male_name_adj_count_df["count"].sum()
f_total_cooc = female_name_adj_count_df["count"].sum()
f_m_total_cooc = m_total_cooc + f_total_cooc

print(
    f"there are {len(intersect_f_m_adj)} adjectives that occur in both a male AND female context"
)
print(
    f"there are {m_total_cooc} occurrences of adjectives in sentences that contained a male character"
)
print(
    f"there are {f_total_cooc} occurrences of adjectives in sentences that contained a female character"
)
print(
    f"there are {f_m_total_cooc} occurrences of adjectives in sentences that contained a female or male character"
)

In [ ]:
# aggregate and sum counts per adjective per gender
# new columns: f_count/m_count -> how many times did an adjective occur next to a female/male character
gender_adj_df = (
    name_adj_count_df.groupby(["adjective", "gender"])["count"]
    .sum()
    .unstack(fill_value=0)
    .rename(columns={"m": "m_count", "f": "f_count"})
    .sort_values(["f_count", "m_count"], ascending=False)
)

MIN_COOC = 2

# filter by minimum occurrences
# gender_adj_df = gender_adj_df[(gender_adj_df["m_count"] > MIN_COOC) & (gender_adj_df["f_count"] >= MIN_COOC)]
gender_adj_df

In [ ]:
# calculate occurrence rates per 1000, normalised frequency, not a percentage
# for each row we calculate how many times this adjectve occurrs per 1000 male/female co-occurrences
# e.g. m_rate = 33.9639 means that this adjective appears 33.96 times per 1000 male co-occurrences
# we divide by total male co-occurrences and multiply by 1000 just to avoid tiny decimals like 0.034
gender_adj_df["m_rate"] = (gender_adj_df["m_count"] / m_total_cooc * 1000).round(4)
gender_adj_df["f_rate"] = (gender_adj_df["f_count"] / f_total_cooc * 1000).round(4)

# calculate direction/magnitude of adjectives
# for each row we calculate how much the female rate is higher / lower per 1000 than the male rate
# e.g. f_minus_m = 56.1701 means that the female rate is 56.17 points per 1000 higher than the male rate -> positive -> this adjective leans female
gender_adj_df["f_minus_m"] = (gender_adj_df["f_rate"] - gender_adj_df["m_rate"]).round(
    4
)

# calculate expected counts
# for each row we calculate how many times we expect this adjective to appear in a female or male context, purely based on hwo common "male"/"female" adjectives are in this corpus
# this is the baseline for G2, G2 compares the observed count against these expected ones
# e.g. E_m = 311.66 means that if there were no gender bias, you'd expect this adjective to appear 311.66 times in a male context
gender_adj_df["E_m"] = (
    m_total_cooc
    * (gender_adj_df["m_count"] + gender_adj_df["f_count"])
    / f_m_total_cooc
)
gender_adj_df["E_f"] = (
    f_total_cooc
    * (gender_adj_df["m_count"] + gender_adj_df["f_count"])
    / f_m_total_cooc
)


# calculate G2 in datafreme directly, use safe version
# https://en.wikipedia.org/wiki/G-test
def safe_ll(observed, expected):
    return np.where(
        (observed > 0) & (expected > 0), observed * np.log(observed / expected), 0.0
    )


gender_adj_df["G2"] = (
    2
    * (
        safe_ll(gender_adj_df["m_count"], gender_adj_df["E_m"])
        + safe_ll(gender_adj_df["f_count"], gender_adj_df["E_f"])
    )
).round(3)

# calculate p-value using Chi-Squared (https://en.wikipedia.org/wiki/Chi-squared_test)
gender_adj_df["p_value"] = chi2.sf(gender_adj_df["G2"], df=1)
# infer direction (is it female or male leaning)
gender_adj_df["direction"] = np.where(
    gender_adj_df["f_rate"] > gender_adj_df["m_rate"], "female", "male"
)

# drop helper columns and sort
gender_adj_df = (
    gender_adj_df
    # .drop(columns=["E_m", "E_f"])
    .sort_values("G2", ascending=False).reset_index()
)
gender_adj_df

#### Adjust for multiple testing error using false discovery rate (fdr) with Benjamini/Hochberg (non-negative)

In [ ]:
# correct p value using benjamini-hochberg fdr-correction with default alpha of 0.05
reject, p_adjusted, _, _ = multipletests(gender_adj_df["p_value"], method="fdr_bh")
gender_adj_df["p_adjusted"] = p_adjusted
gender_adj_df["significant"] = reject

gender_adj_df

In [ ]:
print(f"\nGender-biased adjectives (p < 0.05):")
significant = gender_adj_df[gender_adj_df['significant']]
print(
    "Top male-biased: "
    + ", ".join(significant[significant["direction"] == "male"].head(10)["adjective"])
)
print(
    "Top female-biased: "
    + ", ".join(significant[significant["direction"] == "female"].head(10)["adjective"])
)

---

In [ ]:
# KI generiert
def plot_diverging(gender_comp_df, title, filename, top_n=20):
    if gender_comp_df.empty:
        print(f"Skipping plot '{title}' — no data.")
        return

    male_top = gender_comp_df[gender_comp_df["direction"] == "male"].head(top_n).copy()
    female_top = (
        gender_comp_df[gender_comp_df["direction"] == "female"].head(top_n).copy()
    )

    if male_top.empty and female_top.empty:
        print(f"Skipping plot '{title}' — no male or female skewed adjectives found.")
        return

    male_top["G2_signed"] = male_top["G2"]
    female_top["G2_signed"] = -female_top["G2"]

    plot_df = pd.concat(
        [
            female_top.sort_values("G2_signed"),
            male_top.sort_values("G2_signed"),
        ]
    )

    colors = ["#e07b7b" if v < 0 else "#7baee0" for v in plot_df["G2_signed"]]

    fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.35)))
    ax.barh(
        plot_df["adjective"],
        plot_df["G2_signed"],
        color=colors,
        edgecolor="none",
        alpha=0.85,
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("<- more female  |  G2 keyness  |  more male ->", fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{abs(x):.0f}"))
    male_patch = mpatches.Patch(color="#7baee0", alpha=0.85, label="male characters")
    female_patch = mpatches.Patch(
        color="#e07b7b", alpha=0.85, label="female characters"
    )
    ax.legend(handles=[female_patch, male_patch])
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    # plt.savefig(f"./outputs/{filename}", dpi=150)
    plt.show()
    print(f"Saved {filename}")


plot_diverging(
    significant, "Adjectives near character names by gender", "adj_gender_window.png"
)

---

#### In this part we demonstrate how to view a specific keyword (adj) in its context

In [ ]:
base_forms = sorted(list(reverse_lemma_lookup['nächster']), key=len, reverse=True)
base_forms

In [ ]:
pattern = "|".join(base_forms)
matching_sentences = list(character_rows[character_rows["sentences"].str.contains(pattern, na=False)]['sentences'])
for idx, sentence in enumerate(matching_sentences):
    for bf in base_forms:
        if bf in sentence:
            matching_sentences[idx] = sentence.replace(bf, f'**{bf}**')
            break
for sentence in matching_sentences:
    display(Markdown(sentence))